(content:mapping)=
# Mapping scRNA-seq data and label transfer

The refernce mapping workflow is adapted from the reference mapping with [scvi-tools tutorial](https://docs.scvi-tools.org/en/1.0.0/tutorials/notebooks/scarches_scvi_tools.html) and reflects one of the best performing methods for label transfer according to the living benchmarks of the [Open Problems in Single Cell Analysis](https://openproblems.bio/benchmarks/label_projection?version=v2.0.0) initiative.

There are multiple ways to leverage a reference atlas to transfer cell type labels and perform downstream analysis. Here we show how to avoid the need for downloading the entire scRNA seq data but instead use the trained model used for integration to map a new dataset using (scArches) {cite:p}`lotfollahi2022mapping`.

As an example use case, we map an Crohn's and UC single cell data sets 

In [ ]:
# Load Packages for data display
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy.stats import entropy

import scvi

import os

In [ ]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 300)

In [ ]:
os.getcwd()

In [ ]:
os.chdir('SET PATH TO YOUR WORKING DIRECTORY')

Now we load the query data and ensure the right formatting for reference mapping. The data set we use is from a study on anti-TNF treatment in IBD {cite:p}`thomas2024longitudinal` downloaded from [Zenodo](https://zenodo.org/records/14007626).

In [ ]:
adata_taurus = sc.read_h5ad('data/taurus_data/TAURUS_raw_counts_annotated_final.h5ad')

## Covariates and QC of the query data

In [ ]:
# Quick check of the object
adata_taurus

In [ ]:
# Check if adata.X are counts
adata_taurus.X[:100].toarray()[adata_taurus.X[:100].toarray()>0]

In [ ]:
# check var
adata_taurus.var.head()

We want to make sure, that gene names are set to ensemble IDs, as these are the feature names used for the scVI model.

In [ ]:
# Set ensemble IDs as var_names
adata_taurus.var_names = adata_taurus.var['gene_id']

In [ ]:
# check obs to get a feeling for the metadata available
adata_taurus.obs

As the donor ID was used to correct for batch effects in the HGCA, we select the corresponding `patient` column in the Taurus data as the batch covariate. To better understand potential technical and biological effects, we check how donors are distributed across biological an technical covariates using heatmaps.

In [ ]:
# Function for plotting

def plot_heatmap(adata, category1, cat1_label, category2, cat2_label, title=None):

    # Get crosstab
    ct_table = pd.crosstab(adata.obs[category1], adata.obs[category2])
    width = len(adata.obs[category2].unique())*1.2
    height = len(adata.obs[category1].unique())/4
    
    # Plot heatmap
    plt.figure(figsize=(width, height))

    sns.heatmap(
        ct_table,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar_kws={'label': '# cells'})

    plt.title(title)
    plt.xlabel(cat2_label)
    plt.ylabel(cat1_label)

    plt.show()

In [ ]:
# Function for plotting a heatmap which can be combined with multiple heatmaps in one figure

def plot_heatmap_ax(adata, category1, cat1_label, category2, cat2_label, ax, show_ylabel=False, title=None):
    """Plots a heatmap on a specific matplotlib axis."""
    
    # Get crosstab
    ct_table = pd.crosstab(adata.obs[category1], adata.obs[category2])
    
    # Plot heatmap on the provided axis (ax)
    sns.heatmap(
        ct_table,
        annot=True,
        fmt='d',
        cmap='Blues',
        #cbar_kws={'label': '# cells'},
        ax=ax,
        cbar=False
    )

    # Use ax methods for labels and titles
    if title:
        ax.set_title(title)
    ax.set_xlabel(cat2_label)

    if show_ylabel:
        ax.set_ylabel(cat1_label)
    else:
        ax.set_ylabel('')



In [ ]:
cats= ['Disease', 'Site', 'Inflammation']

fig, axes = plt.subplots(
    nrows=1, ncols=3,
    figsize=(14, 9),
    gridspec_kw={'width_ratios': [2,3.5,2]},
    sharey=True,
    )

for i,cat in enumerate(cats):

    plot_heatmap_ax(
        adata=adata_taurus, 
        category1='Patient', cat1_label='Patient', 
        category2=cat, cat2_label=cat, 
        title='', 
        ax=axes[i]
    )

# 5. Prevent labels from overlapping and display the final figure
plt.tight_layout()
plt.subplots_adjust(wspace=0.2)
plt.show()

Now let's also check the technical batch

In [ ]:
# Plot for processing batch
plot_heatmap(
    adata_taurus,
    category1='Patient',
    cat1_label='Patient',
    category2='Batch',
    cat2_label='Processing batch'
    
)


A few interesting aspects we observe is that there are inflammed and non inflammed samples for each patient, all healthy donors were processed in the same processing batch and there is an overrepresentation of UC samples among the biopsies takes from the Sigmoid. None of these observations might be an issue, but it is good to keep those trends in mind for downstream analysis.

For mapping the data to the HGCA, we need to reduce the object to the features used for the integration of the HGCA. To allow analysis of all genes later on, let's save the full object.

In [ ]:
# Save copy with full gene space
adata_taurus_full = adata_taurus.copy()

In [ ]:
# In case you want to reduce the data set for faster processing of an example, reduce to only CD cases and remove UC.
#adata_taurus = adata_taurus[adata_taurus.obs['Disease']!= 'UC', :].copy()

In [ ]:
adata_taurus

Next, we check the quality of each sample and look for questionable QC metric distributions. Note that epithelial cells often show higher fractions of transcripts from mitochondrial genes which does not necessarily indicate samples failure.

In [ ]:
# Sanity check of QC metrics
sc.set_figure_params(figsize=(34, 5))
sc.pl.violin(adata_taurus, ['pct_counts_mt'], groupby='sample_id', rotation=90)
plt.show()

In [ ]:
# Sanity check of QC metrics
sc.set_figure_params(figsize=(34, 5))
sc.pl.violin(adata_taurus, ['total_counts'], groupby='sample_id', rotation=90 , log=True)
plt.show()

In [ ]:
# Sanity check of QC metrics
sc.set_figure_params(figsize=(34, 5))

ax = sc.pl.violin(adata_taurus, ['n_genes_by_counts'], groupby='sample_id', rotation=90, show=False)
ax.set_ylim(0, 3000)

plt.show()

We can see that a minimum of 500 genes per cell was used to filter out low quality cells and a maximum of 60% counts of mitochondrial genes. We indeed see a few samples with a stretched QC metric distributions (total counts & number of genes per cell) combined with higher fractions of mitochondiral counts (e.g. the third sample). This we should keep in mind, in case those samples appear to map with lower certainty compared to the other samples. As you might have seen from the general characteristics of scRNA data the higher fraction of mitochondrial transcripts can also be driven by a higher fraction of epithelial cells in this specific sample.

## Lineage prediction

The highest quality integrations of the HGCA are lineage specific integrated objects. Hence, we would like to split up the Taurus data into the four linage objects. To do so, we could map the author labels to the lineages, but to demonstrate how a novel dataset without cell type labels could be annotated, we map and transfer cell type labels using the full HGCA objects first, followed by splitting the data and mapping it a second time to the lineage objects for high resolution cell type prediction.

In [ ]:
# Load the trained integration model
scanvi_full_model = scvi.model.SCANVI.load('hca-gut-atlas-extension/data/pipeline_data/models/postCAP_models/scanvi_full_gca_concat_hgca_celltype_v1_minified.pt')


In [ ]:
scanvi_full_model # Might show minified?: False, even though the model was saved with minified=True. Might be a bug in the current scvi version.

In [ ]:
# Check if genes are present
scvi.model.SCANVI.prepare_query_anndata(adata_taurus, scanvi_full_model)

In [ ]:
# Ensure batch covariate is available by adding an obs column of the same name as the batch covariate in the HGCA.
adata_taurus.obs['donor_id'] = adata_taurus.obs['Patient'].astype('category')

To show label transfer to a new dataset, we set all cell type labels to `"Unknown"`, matching the `unlabeled_category: Unknown` in the scanvi model.

In [ ]:
# Set Unknown cell type 
adata_taurus.obs['hgca_celltype_v1'] = "Unknown"

In [ ]:
# Load the query data
vae_q = scvi.model.SCANVI.load_query_data(
    adata_taurus,
    scanvi_full_model
)

We are ready for the actual mapping process based on (scArches) {cite:p}`lotfollahi2022mapping`. If possible run this on a GPU tp speed it up.

In [ ]:
# Train the query model
vae_q.train(
    max_epochs=60,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

Saving the model to allow analysis later on.

In [ ]:
# save scANVI model with taurus
vae_q.save("hca-gut-atlas-extension/data/pipeline_data/models/postCAP_models/scanvi_full_gca_concat_hgca_celltype_v1_taurus_map.pt")

In [ ]:
# re-load scANVI model with taurus
vae_q = scvi.model.SCANVI.load(
    "hca-gut-atlas-extension/data/pipeline_data/models/postCAP_models/scanvi_full_gca_concat_hgca_celltype_v1_taurus_map.pt",
    adata = adata_taurus
    )


Let's get the latent representation of the mapped data, and predict the cell types. The cell type level (e.i. resolution) was defined during the training process.

In [ ]:
# Get the latent representation
adata_taurus.obsm["X_scANVI_across_lin"] = vae_q.get_latent_representation()

In [ ]:
# Predic the labels
adata_taurus.obs["predictions"] = vae_q.predict()

Since we do have cell types labels from the original publication, we can do a quick comparison of the author labels and the predicted cell type labels using the cross lineage HGCA integration model. Later we can compare these labels als with the lineage specific labels. The author annotations are stored in the `final_analysis` columns in the `adata.obs`.

In [ ]:
df =pd.crosstab(adata_taurus.obs['final_analysis'], adata_taurus.obs['predictions'], margins=True, margins_name='Total') # Incl. margins to get the total number of cells per label

In [ ]:
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

row_tot = df.loc[:, 'Total'].drop('Total')      # cells per final_analysis label
col_tot = df.loc['Total', :].drop('Total')      # cells per prediction label
mat     = df.drop(index='Total', columns='Total')

NORM = 'row'          # 'row' -> % of each true label; 'col' -> % of each prediction
norm = (mat.div(mat.sum(1), axis=0) if NORM == 'row'
        else mat.div(mat.sum(0), axis=1)) * 100

# optional: order columns so the dominant mapping runs along the diagonal
order = norm.idxmax(1).map({c: i for i, c in enumerate(norm.columns)}).sort_values().index
norm, mat, row_tot = norm.loc[order], mat.loc[order], row_tot.loc[order]


n_row, n_col = norm.shape


In [ ]:
def plot_agreement_heatmap(df, NORM='row'):

    # ---------------------------------------------------------------- layout
    fig = plt.figure(figsize=(0.17 * n_col + 4.5, 0.17 * n_row + 4.5))
    gs = GridSpec(2, 3, figure=fig,
                width_ratios=[n_col, 9, 1.2], height_ratios=[9, n_row],
                wspace=0.02, hspace=0.02)

    ax_t  = fig.add_subplot(gs[0, 0])                  # column totals, on top
    ax_hm = fig.add_subplot(gs[1, 0], sharex=ax_t)     # heatmap
    ax_r  = fig.add_subplot(gs[1, 1], sharey=ax_hm)    # row totals
    ax_cb = fig.add_subplot(gs[1, 2])

    # ---------------------------------------------------------------- heatmap
    cmap = plt.get_cmap('Blues').copy()
    cmap.set_bad('#f5f5f5')                      # zeros read as empty, not "low"
    data = np.ma.masked_where(norm.values == 0, norm.values)

    mesh = ax_hm.pcolormesh(data, cmap=cmap, vmin=0, vmax=100,
                            edgecolors='white', linewidth=0.3)

    ax_hm.set_xlim(0, n_col); ax_hm.set_ylim(n_row, 0)
    ax_hm.set_yticks(np.arange(n_row) + .5, norm.index, fontsize=6)
    ax_hm.set_xticks(np.arange(n_col) + .5, norm.columns, rotation=90, fontsize=6)
    ax_hm.set_ylabel('final_analysis', fontsize=9)
    ax_hm.set_xlabel('predictions', fontsize=9)
    for s in ax_hm.spines.values():
        s.set_visible(False)

    cb = fig.colorbar(mesh, cax=ax_cb)
    cb.set_label(f'% of {NORM}', fontsize=8)
    cb.ax.tick_params(labelsize=7)
    cb.outline.set_visible(False)

    # ---------------------------------------------------------------- top bars
    ax_t.bar(np.arange(n_col) + .5, col_tot.values, width=0.8,
            color='#94a3b8', linewidth=0)
    ax_t.set_yscale('log')
    ax_t.set_ylabel('n cells', fontsize=8)
    ax_t.tick_params(axis='x', bottom=False, labelbottom=False)   # labels live on the heatmap
    ax_t.tick_params(axis='y', labelsize=7)
    ax_t.grid(axis='y', color='white', lw=0.8)
    ax_t.set_axisbelow(True)
    for s in ax_t.spines.values():
        s.set_visible(False)

    # ---------------------------------------------------------------- right bars
    ax_r.barh(np.arange(n_row) + .5, row_tot.values, height=0.8,
            color='#94a3b8', linewidth=0)
    ax_r.set_xscale('log')
    ax_r.set_xlabel('n cells', fontsize=8)
    ax_r.tick_params(axis='y', left=False, labelleft=False)
    ax_r.tick_params(axis='x', labelsize=7)
    ax_r.grid(axis='x', color='white', lw=0.8)
    ax_r.set_axisbelow(True)
    for s in ax_r.spines.values():
        s.set_visible(False)

    fig.suptitle(f'Label transfer: final_analysis vs predictions  (row-normalised, n={mat.values.sum():,})',
                fontsize=11, y=0.995)
    #fig.savefig('label_transfer_heatmap.png', dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
plot_agreement_heatmap(df)

The heatmap shows how most cell types have distinct matches. Let's plot them on the UMAP and then assign the linaege labels used to split up the object into the lineage objects.

In [ ]:
print('Neighbors...')
sc.pp.neighbors(adata_taurus , use_rep="X_scANVI_across_lin")
print('UMAP...')
sc.tl.umap(adata_taurus)

In [ ]:
sc.set_figure_params(figsize=(12, 12))
sc.pl.umap(adata_taurus, color='predictions', title='Predicted labels',size=1)

We can appreciate that many cell types could be labelled and the latent representation of the Taurus data seems to cover a large spectrum of cell types.
Let's map the fine grained labels to the linage level to split up the data for the mapping to the lineage spacific objects of the HGCA.

In [ ]:
# Load cell type level mapping reference

cell_type_ref_df = pd.read_csv('hca-gut-atlas-tutorial/data/hgca_cell_type_reference.tsv', sep='\t')
coarse_to_lineage_map = dict(zip(cell_type_ref_df['hgca_celltype_v1'], cell_type_ref_df['hgca_celltype_level1']))


In [ ]:
adata_taurus.obs['predicted_lineage'] = adata_taurus.obs['predictions'].map(coarse_to_lineage_map)

In [ ]:
# TODO: Save data with full hca predicted labels
adata_taurus.write_h5ad("hca-gut-atlas-downstream/data/taurus_data/taurus_hgca_full_mapped_hvg.h5ad")

## Map per lineage objects

In [ ]:
adata_taurus=sc.read_h5ad("hca-gut-atlas-downstream/data/taurus_data/taurus_hgca_full_mapped_hvg.h5ad")

In [ ]:
adata_taurus

In [ ]:
sc.pl.umap(adata_taurus, color='predicted_lineage')

In [ ]:
pd.crosstab(adata_taurus.obs['final_analysis'], adata_taurus.obs['predicted_lineage'])

In [ ]:
lin_list =['myeloid', 'epithelial', 'lymphoid', 'stroma']

In [ ]:
# Split into lineage objects

adata_taurus_myeloid = adata_taurus[adata_taurus.obs['predicted_lineage']=='Myeloid',:].copy()
adata_taurus_epi = adata_taurus[adata_taurus.obs['predicted_lineage']=='Epithelial',:].copy()
adata_taurus_lymph = adata_taurus[adata_taurus.obs['predicted_lineage']=='Lymphoid',:].copy()
adata_taurus_stroma = adata_taurus[adata_taurus.obs['predicted_lineage']=='Stromal',:].copy()

In [ ]:
adata_taurus_myeloid.shape

In [ ]:
adata_taurus_epi.shape

In [ ]:
adata_taurus_lymph.shape

In [ ]:
adata_taurus_stroma.shape

In [ ]:
taurus_adata_list=[adata_taurus_myeloid, adata_taurus_epi, adata_taurus_lymph, adata_taurus_stroma]

In [ ]:
i=0

In [ ]:
# Load the trained integration model
scanvi_lin_model = scvi.model.SCANVI.load("hca-gut-atlas-extension/data/pipeline_data/models/postCAP_models/scanvi_"+str(lin_list[i])+"_hgca_celltype_v1_minified.pt")



In [ ]:
scanvi_lin_model # Might show minified?: False, even though the model was saved with minified=True. Might be a bug in the current scvi version.

In [ ]:
# Check if genes are present
scvi.model.SCANVI.prepare_query_anndata(taurus_adata_list[i], scanvi_lin_model)

In [ ]:
# Ensure batch covariate is available by adding an obs column of the same name as the batch covariate in the HGCA.
taurus_adata_list[i].obs['donor_id'] = taurus_adata_list[i].obs['Patient'].astype('category')

To show label transfer to a new dataset, we set all cell type labels to `"Unknown"`, matching the `unlabeled_category: Unknown` in the scanvi model.

In [ ]:
# Set Unknown cell type 
taurus_adata_list[i].obs['hgca_celltype_v1'] = "Unknown"

In [ ]:
# Load the query data
vae_q = scvi.model.SCANVI.load_query_data(
    taurus_adata_list[i],
    scanvi_lin_model
)

We are ready for the actual mapping process based on (scArches) {cite:p}`lotfollahi2022mapping`. If possible run this on a GPU tp speed it up.

In [ ]:
# Train the query model
vae_q.train(
    max_epochs=60,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

Saving the model to allow analysis later on.

In [ ]:
# save scANVI model with taurus
vae_q.save("hca-gut-atlas-extension/data/pipeline_data/models/postCAP_models/scanvi_"+str(lin_list[i])+"_hgca_celltype_v1_taurus_map_v2.pt")

In [ ]:
# re-load scANVI model with taurus
vae_q = scvi.model.SCANVI.load(
    "hca-gut-atlas-extension/data/pipeline_data/models/postCAP_models/scanvi_"+str(lin_list[i])+"_hgca_celltype_v1_taurus_map_v2.pt",
    adata = taurus_adata_list[i]
    )



Let's get the latent representation of the mapped data, and predict the cell types. The cell type level (e.i. resolution) was defined during the training process.

In [ ]:
# Get the latent representation
taurus_adata_list[i].obsm["X_scVI_"+str(lin_list[i])+"_lin"] = vae_q.get_latent_representation()

In [ ]:
# Predic the labels
taurus_adata_list[i].obs["predictions_lineage"] = vae_q.predict()

In [ ]:
taurus_adata_list[i].obs["predictions"].value_counts()

In [ ]:
taurus_adata_list[i]

The heatmap shows how most cell types have distinct matches. Let's plot them on the UMAP and then assign the linaege labels used to split up the object into the lineage objects.

In [ ]:
print('Neighbors...')
sc.pp.neighbors(taurus_adata_list[i] , use_rep="X_scVI_"+str(lin_list[i])+"_lin")
print('UMAP...')
sc.tl.umap(taurus_adata_list[i])

In [ ]:
sc.set_figure_params(figsize=(12, 12))
sc.pl.umap(taurus_adata_list[i], color='predictions_lineage', title='Predicted labels on lineage objects',size=1)

In [ ]:
taurus_adata_list[i].write_h5ad("hca-gut-atlas-downstream/data/taurus_data/taurus_hgca_"+lin_list[i]+"_mapped_hvg.h5ad")

## Load mapped per lineage data

In [ ]:
lin_list =['myeloid', 'epithelial', 'lymphoid', 'stroma']

In [ ]:
taurus_adata_list = []

for i in [0,1,2,3]:
    adata = sc.read_h5ad("hca-gut-atlas-downstream/data/taurus_data/taurus_hgca_"+lin_list[i]+"_mapped_hvg.h5ad")
    taurus_adata_list.append(adata)
    

### Add certainty of label predictions

In [ ]:
# Re-load scANVI model with taurus

for i in [0,1,2,3]:

    vae_q = scvi.model.SCANVI.load(
        "hca-gut-atlas-extension/data/pipeline_data/models/postCAP_models/scanvi_"+str(lin_list[i])+"_hgca_celltype_v1_taurus_map.pt",
        adata = taurus_adata_list[i]
        )

    soft = vae_q.predict(soft=True)
    taurus_adata_list[i].obs["pred_prob"]    = soft.max(axis=1).values
    taurus_adata_list[i].obs["pred_entropy"] = entropy(soft.values, axis=1)     
    taurus_adata_list[i].obsm["scanvi_soft"] = soft

    

In [ ]:
# Save latent space with obs_names
for i,lin in enumerate (lin_list):
    ct_df = pd.DataFrame(taurus_adata_list[0].obs.loc[:,['predictions', 'pred_prob', 'pred_entropy']], index=taurus_adata_list[0].obs_names)
    ct_df.to_pickle('hca-gut-atlas-downstream/data/taurus_data/taurus_'+str(lin)+'_mapped_celltypes.pkl')

In [ ]:
ct_df = pd.read_pickle('hca-gut-atlas-downstream/data/taurus_data/taurus_'+str(lin)+'_mapped_celltypes.pkl')

In [ ]:
for i in [0,1,2,3]:
    sc.pl.umap(taurus_adata_list[i], color=['pred_prob', 'pred_entropy', 'predictions'], ncols=2)

In [ ]:
for i in [0,1,2,3]:
    df =pd.crosstab(taurus_adata_list[i].obs['final_analysis'], taurus_adata_list[i].obs['predictions'], margins=True, margins_name='Total') # Incl. margins to get the total number of cells per label
    row_tot = df.loc[:, 'Total'].drop('Total')      # cells per final_analysis label
    col_tot = df.loc['Total', :].drop('Total')      # cells per prediction label
    mat     = df.drop(index='Total', columns='Total')

    NORM = 'col'          # 'row' -> % of each true label; 'col' -> % of each prediction
    norm = (mat.div(mat.sum(1), axis=0) if NORM == 'row'
            else mat.div(mat.sum(0), axis=1)) * 100

    # optional: order columns so the dominant mapping runs along the diagonal
    order = norm.idxmax(1).map({c: i for i, c in enumerate(norm.columns)}).sort_values().index
    norm, mat, row_tot = norm.loc[order], mat.loc[order], row_tot.loc[order]

    n_row, n_col = norm.shape

    plot_agreement_heatmap(df)

In [ ]:
# Save cell type labels and obs names only
for i in [0,1,2,3]:
    ct_df = pd.DataFrame(taurus_adata_list[i].obs['predictions'], index=taurus_adata_list[i].obs_names)
    ct_df.to_pickle("hca-gut-atlas-downstream/data/taurus_data/taurus_"+lin_list[i]+"_mapped_celltypes.pkl")

### Concatenating reference and query

In [ ]:
os.getcwd()

In [ ]:
hgca_lin_adata_list=[]

for i,lin in enumerate(lin_list):
    adata = sc.read_h5ad('hca-gut-atlas-extension/data/concat_objects/adata_after_cap/'+str(lin)+'_incl_scvi.h5ad')
    
    scanvi_latent = pd.read_pickle('hca-gut-atlas-extension/data/concat_objects/adata_after_cap/'+str(lin)+'_scANVI_hgca_celltype_v1_latent.pkl')
    scanvi_umap = pd.read_pickle('hca-gut-atlas-extension/data/concat_objects/adata_after_cap/'+str(lin)+'_scANVI_hgca_celltype_v1_latent_umap.pkl')

    adata.obsm['X_scANVI_'+str(lin)+'_lin'] = scanvi_latent.values
    adata.obsm['X_umap_scANVI_'+str(lin)+'_lin'] = scanvi_umap.values

    hgca_lin_adata_list.append(adata)
    


In [ ]:
taurus_adata_list[1]

In [ ]:
# Standardize most important slots names to keep them during concatenation

#Set scVI to scANVI in obsm (was incorrectly labelled)
for i,lin in enumerate(lin_list):
    taurus_adata_list[i].obsm['X_scANVI_'+str(lin)+'_lin'] = taurus_adata_list[i].obsm['X_scVI_'+str(lin)+'_lin'].copy()
    del taurus_adata_list[i].obsm['X_scVI_'+str(lin)+'_lin']


In [ ]:
# Standardize most important slots names to keep them during concatenation

#Set scVI to scANVI in obsm (was incorrectly labelled)
for i,lin in enumerate(lin_list):
    taurus_adata_list[i].obsm['X_umap_scANVI_'+str(lin)+'_lin'] = taurus_adata_list[i].obsm['X_umap'].copy()


In [ ]:


# change column name of predicted labels from full gca object
for i,lin in enumerate(lin_list):
    taurus_adata_list[i].obs = taurus_adata_list[i].obs.rename(columns={'hgca_celltype_v1':'hgca_celltype_v1_full'})

col_names_map = {
    'Disease':'disease_ontology_term',
    'Site':'institute',
    'Gender':'sex_ontology_term',
    'Batch':'author_batch_notes',
    'doublet_scores':'doublet_score',
    'n_genes_by_counts':'n_genes',
    'total_counts':'n_counts',
    'final_analysis':'author_cell_type',
    'predicted_lineage':'hgca_celltype_level1',
    'predictions':'hgca_celltype_v1'
}

for i,lin in enumerate(lin_list):
    taurus_adata_list[i].obs = taurus_adata_list[i].obs.rename(columns=col_names_map)



In [ ]:
hgca_lin_adata_list[0]

In [ ]:
taurus_adata_list[0]

In [ ]:
adata_myeloid_full = ad.concat([taurus_adata_list[0], hgca_lin_adata_list[0]], label="subset")
adata_epi_full = ad.concat([taurus_adata_list[1], hgca_lin_adata_list[1]], label="subset")
adata_lymph_full = ad.concat([taurus_adata_list[2], hgca_lin_adata_list[2]], label="subset")
adata_stromal_full = ad.concat([taurus_adata_list[3], hgca_lin_adata_list[3]], label="subset")

In [ ]:
adata_myeloid_full.obs["subset"] = adata_myeloid_full.obs["subset"].cat.rename_categories(
    ["Query", "Reference"]
)
adata_myeloid_full

In [ ]:
adata_epi_full.obs["subset"] = adata_epi_full.obs["subset"].cat.rename_categories(
    ["Query", "Reference"]
)
adata_epi_full

In [ ]:
adata_lymph_full.obs["subset"] = adata_lymph_full.obs["subset"].cat.rename_categories(
    ["Query", "Reference"]
)
adata_lymph_full

In [ ]:
adata_stromal_full.obs["subset"] = adata_stromal_full.obs["subset"].cat.rename_categories(
    ["Query", "Reference"]
)
adata_stromal_full

In [ ]:
sc.pp.neighbors(adata_myeloid_full, use_rep='X_scANVI_'+str(lin_list[0])+'_lin')
sc.tl.umap(adata_myeloid_full)

In [ ]:
sc.pp.neighbors(adata_epi_full, use_rep='X_scANVI_'+str(lin_list[1])+'_lin')
sc.tl.umap(adata_epi_full)

In [ ]:
sc.pp.neighbors(adata_lymph_full, use_rep='X_scANVI_'+str(lin_list[2])+'_lin')
sc.tl.umap(adata_lymph_full)

In [ ]:
sc.pp.neighbors(adata_stromal_full, use_rep='X_scANVI_'+str(lin_list[3])+'_lin')
sc.tl.umap(adata_stromal_full)

In [ ]:
sc.pl.umap(adata_myeloid_full, color=['subset', 'hgca_celltype_v1', 'disease_ontology_term', 'institute'], ncols=2, title='Myeloid cells mapped to HGCA', wspace=0.6)

In [ ]:
sc.pl.umap(adata_epi_full, color=['subset', 'hgca_celltype_v1'], title='Epithelial cells mapped to HGCA', wspace=0.2)

In [ ]:
sc.pl.umap(adata_lymph_full, color=['subset', 'hgca_celltype_v1'], title='Lymphoid cells mapped to HGCA', wspace=0.2)

In [ ]:
sc.pl.umap(adata_stromal_full, color=['subset', 'hgca_celltype_v1'], title='Stromal cells mapped to HGCA', wspace=0.2)

In [ ]:
# Save taurus/gca concat with umap...
adata_myeloid_full.write_h5ad("hca-gut-atlas-downstream/data/taurus_data/taurus_hgca_"+lin_list[0]+"_hgca_concat.h5ad")
adata_epi_full.write_h5ad("hca-gut-atlas-downstream/data/taurus_data/taurus_hgca_"+lin_list[1]+"_hgca_concat.h5ad")
adata_lymph_full.write_h5ad("hca-gut-atlas-downstream/data/taurus_data/taurus_hgca_"+lin_list[2]+"_hgca_concat.h5ad")
adata_stromal_full.write_h5ad("hca-gut-atlas-downstream/data/taurus_data/taurus_hgca_"+lin_list[3]+"_hgca_concat.h5ad")


## Identify disease populations

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def weighted_knn_uncertainty(ref_emb, ref_labels, query_emb, n_neighbors=50):
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights="distance")
    knn.fit(ref_emb, ref_labels)
    proba = knn.predict_proba(query_emb)          # weighted neighbour vote
    classes = knn.classes_
    pred_idx = proba.argmax(1)
    pred = classes[pred_idx]
    conf = proba[np.arange(len(proba)), pred_idx]  # weighted agreement
    return pred, 1.0 - conf                        # (label, uncertainty in [0,1])

# ref = sc.read_h5ad("hgca_<lin>_reference.h5ad")   # reference in same latent
# knn_pred, knn_uncert = weighted_knn_uncertainty(
#     ref.obsm[LATENT], ref.obs["hgca_celltype_v1"].values,
#     adata.obsm[LATENT], n_neighbors=50)
# adata.obs["knn_pred"]   = knn_pred
# adata.obs["knn_uncert"] = knn_uncert
# adata.obs["knn_pred_masked"] = np.where(knn_uncert > 0.2, "Unknown", knn_pred)